# Module 12 - 실습 2: 서브그래프 패턴

## 학습 목표
- 서브그래프를 독립적으로 정의하고 빌드할 수 있다
- 메인 그래프에서 서브그래프를 노드로 사용할 수 있다
- 부모-자식 그래프 간 상태 매핑을 이해한다

## 참조 자료
- 학습 문서: `docs/part5-advanced/12-langgraph-advanced.md` (섹션 2.2)

## 개념 요약

```
[메인 그래프]
[A] → [서브그래프] → [C]
           │
      [X] → [Y] → [Z]

상태 매핑:
메인 text 필드 → 서브그래프 text 필드 (이름이 같으면 자동 매핑)
서브그래프 keywords/summary → 메인 keywords/summary
```

서브그래프는 자신의 상태 필드만 접근하므로 **상태 격리**가 됩니다.

In [ ]:
import sys
sys.path.insert(0, '..')

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from common.fake_llm import FakeLLM

print("서브그래프 실습 준비 완료")

## 실습 1: 서브그래프 상태 및 노드 정의

분석 서브그래프의 상태(`AnalysisSubState`)와 노드들을 정의하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: AnalysisSubState에는 text, keywords, summary, sentiment 필드가 있음
# 힌트 2: 각 노드는 해당 필드만 업데이트하는 dict를 반환

class AnalysisSubState(TypedDict):
    """분석 서브그래프의 격리된 상태."""
    text: str
    keywords: list[str] | None
    summary: str | None
    sentiment: str | None

def extract_keywords_node(state: AnalysisSubState) -> dict:
    """키워드를 추출합니다."""
    # TODO: text에서 길이 4 이상의 단어를 최대 5개 추출
    pass

def summarize_node(state: AnalysisSubState) -> dict:
    """텍스트를 요약합니다."""
    # TODO: text의 앞 100자 + "..." (100자 초과 시)
    pass

def analyze_sentiment_node(state: AnalysisSubState) -> dict:
    """감성을 분석합니다."""
    # TODO: 긍정/부정/중립 키워드로 sentiment 분류
    pass

## 실습 2: 서브그래프 빌드

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트: extract_keywords → summarize → analyze_sentiment → END

def build_analysis_subgraph():
    """분석 서브그래프를 빌드합니다."""
    # TODO: StateGraph(AnalysisSubState)로 서브그래프 구성
    pass

# 서브그래프 독립 테스트
sub_app = build_analysis_subgraph()
sub_result = sub_app.invoke({
    "text": "이 제품은 정말 훌륭합니다. 성능이 좋고 만족스럽습니다.",
    "keywords": None,
    "summary": None,
    "sentiment": None,
})
print(f"키워드: {sub_result.get('keywords')}")
print(f"요약: {sub_result.get('summary')}")
print(f"감성: {sub_result.get('sentiment')}")

In [ ]:
# 검증 셀 - 서브그래프 독립 테스트
assert sub_result.get("keywords") is not None, "keywords가 없습니다"
assert sub_result.get("summary") is not None, "summary가 없습니다"
assert sub_result.get("sentiment") in ["positive", "negative", "neutral"], \
    f"sentiment가 올바르지 않습니다: {sub_result.get('sentiment')}"
print("✅ 실습 2 완료! 서브그래프가 독립적으로 동작합니다.")

## 실습 3: 메인 그래프 정의 및 서브그래프 통합

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: MainState에는 document_id, raw_text, cleaned_text, text, keywords, summary, sentiment, report 필드
# 힌트 2: text 필드는 서브그래프 text와 매핑됨 (이름이 같으므로 자동 매핑)
# 힌트 3: graph.add_node("analyze", analysis_subgraph) 로 서브그래프를 노드로 등록

class MainState(TypedDict):
    document_id: str
    raw_text: str
    cleaned_text: str | None
    text: str              # 서브그래프 입력용
    keywords: list[str] | None
    summary: str | None
    sentiment: str | None
    report: str | None

def preprocess_node(state: MainState) -> dict:
    """텍스트를 전처리합니다."""
    # TODO: raw_text를 정리하고 text 필드에 설정
    pass

def generate_report_node(state: MainState) -> dict:
    """최종 리포트를 생성합니다."""
    # TODO: keywords, summary, sentiment를 조합하여 report 생성
    pass

def build_main_graph():
    """서브그래프가 포함된 메인 그래프를 구성합니다."""
    # TODO: 구현
    pass

app = build_main_graph()

In [ ]:
# 메인 그래프 실행
result = app.invoke(
    {
        "document_id": "DOC-001",
        "raw_text": "이 제품은 정말 훌륭합니다. 성능이 좋고 디자인이 만족스럽습니다.",
        "cleaned_text": None,
        "text": "",
        "keywords": None,
        "summary": None,
        "sentiment": None,
        "report": None,
    },
    config={"configurable": {"thread_id": "doc-001"}},
)

print(result.get("report", "리포트가 없습니다"))

In [ ]:
# 검증 셀
assert result.get("report") is not None, "report가 없습니다"
assert result.get("keywords") is not None, "서브그래프 keywords가 메인에 매핑되지 않았습니다"
assert result.get("sentiment") is not None, "서브그래프 sentiment가 메인에 매핑되지 않았습니다"
print(f"✅ 실습 3 완료! 서브그래프가 메인 그래프에 통합되었습니다. (감성: {result['sentiment']})")

## 정리

이번 실습에서 배운 내용:
1. 서브그래프는 독립적인 StateGraph로 정의하고 `compile()`하면 노드처럼 사용할 수 있다
2. 이름이 같은 필드를 통해 메인-서브 그래프 간 상태가 자동 매핑된다
3. 서브그래프를 독립적으로 테스트하고 재사용할 수 있다

## 다음 모듈
- **실습 3**: `03_parallel_send.ipynb` - Send API 병렬 실행

## 보너스: 동적 그래프 구성 (Configurable)

실행 시점에 config 값에 따라 노드를 추가하거나 제거하는 동적 그래프를 구성할 수 있습니다.

예를 들어, `enable_validation` 옵션이 `True`일 때만 검증 노드를 포함시키는 식으로
그래프를 런타임에 유연하게 빌드할 수 있습니다.

In [ ]:
# TODO: config에 따라 노드를 추가/제거하는 동적 그래프를 만드세요
# 힌트 1: build_graph(config) 함수를 만들어 config 값에 따라 분기
# 힌트 2: config.get("enable_validation", False) 로 검증 노드 추가 여부 결정
# 힌트 3: if enable_validation: graph.add_node("validate", validate_fn)

def build_dynamic_graph(config: dict):
    """config에 따라 동적으로 그래프를 구성합니다."""
    # TODO: 아래를 구현하세요
    pass

# 검증 노드 없이 실행
graph_without_validation = build_dynamic_graph({"enable_validation": False})

# 검증 노드 포함하여 실행
graph_with_validation = build_dynamic_graph({"enable_validation": True})

print(f"검증 없음: {graph_without_validation}")
print(f"검증 포함: {graph_with_validation}")

In [ ]:
# 검증 셀
assert graph_without_validation is not None, "build_dynamic_graph({'enable_validation': False})를 구현하세요"
assert graph_with_validation is not None, "build_dynamic_graph({'enable_validation': True})를 구현하세요"

# 두 그래프가 서로 다른 노드 구성을 가져야 함
nodes_without = set(graph_without_validation.get_graph().nodes.keys())
nodes_with = set(graph_with_validation.get_graph().nodes.keys())
assert "validate" not in nodes_without, "'validate' 노드는 enable_validation=False 일 때 없어야 합니다"
assert "validate" in nodes_with, "'validate' 노드는 enable_validation=True 일 때 있어야 합니다"

print("✅ 보너스 완료! 동적 그래프 구성이 올바르게 동작합니다.")